# Accessing Data

In [1]:
#This gets us into the right directory from home, in order to run the import python script from Mat

#sys = system (module), gives information and control over python interpreter/terminal itself
import sys
sys.path.append("/home/565/pv3484/aus_substation_electricity")

#% is a magic command, special shortcut command that lets you control/interact with notebook environment (ie. gives control of terminal without writing full python code)
%cd /home/565/pv3484/aus_substation_electricity

!pwd

/home/565/pv3484/aus_substation_electricity
/home/565/pv3484/aus_substation_electricity


In [2]:
#This section imports the substations that Mat put together

%run /home/565/pv3484/aus_substation_electricity/import_substation.py

processing nsw substations for ['ausgrid'] from None to None
ausgrid
following columns in demand are not in info index:
['MT_HU', 'SI_NO']
removing these columns from demand
number of substations in ausgrid substation info: 134
number of substations in ausgrid substation data: 132
following sites match selection criteria:
               energy_asset          Name  Area  Dwellings  Persons  Residential  Commercial  Industrial  Primary Production  Education  \
ID                                                                                                                                        
BLAKE         AG_BLAKEHURST    Blakehurst     7      10081    28521        0.850       0.005       0.021               0.000      0.022   
PUNCH          AG_PUNCHBOWL     Punchbowl     9      17514    50395        0.826       0.048       0.038               0.000      0.025   
MEADO         AG_MEADOWBANK    Meadowbank    15      22420    56948        0.825       0.023       0.015               0

## Holiday Function

In [3]:
import pandas as pd
from datetime import date, timedelta
from dateutil.easter import easter

#Monarch's Birthday
def second_monday_of_june(y):
    """Return the date of the second Monday in June for year y."""
    june = pd.date_range(start=f"{y}-06-01", end=f"{y}-06-30", freq="D")
    mondays = june[june.weekday == 0]   # Monday = 0
    return mondays[1]                   # second Monday



# Define all national public holidays (including moving ones like Easter)
HOLIDAYS_VIC = {
    "New Year's Day": lambda y: pd.Timestamp(f"{y}-01-01"),
    "Australia Day": lambda y: pd.Timestamp(f"{y}-01-26"),
    "Good Friday": lambda y: pd.Timestamp(easter(y)) - pd.Timedelta(days=2),
    "Easter Saturday": lambda y: pd.Timestamp(easter(y)) - pd.Timedelta(days=1),
    "Easter Sunday": lambda y: pd.Timestamp(easter(y)),
    "Easter Monday": lambda y: pd.Timestamp(easter(y)) + pd.Timedelta(days=1),
    "ANZAC Day": lambda y: pd.Timestamp(f"{y}-04-25"),
    "Monarch's Birthday": lambda y: second_monday_of_june(y),
    "Christmas Day": lambda y: pd.Timestamp(f"{y}-12-25"),
    "Boxing Day": lambda y: pd.Timestamp(f"{y}-12-26"),
}

HOLIDAY_GROUPS = {
    "Good Friday": "Easter Long Weekend",
    "Easter Saturday": "Easter Long Weekend",
    "Easter Sunday": "Easter Long Weekend",
    "Easter Monday": "Easter Long Weekend",

    "Christmas Day": "Christmas and Boxing Day",
    "Boxing Day": "Christmas and Boxing Day",
}


In [ ]:
print(HOLIDAYS_VIC)


# Lat/lon information

In [4]:
from geopy.geocoders import Nominatim
import pandas as pd
import time

# Initialize geocoder
geolocator = Nominatim(user_agent="sydney_demand_mapper")

def get_coords(place):
    """Return (lat, lon) for a suburb name, or (None, None) if not found."""
    try:
        loc = geolocator.geocode(f"{place}, New South Wales, Australia")
        if loc:
            return loc.latitude, loc.longitude
    except Exception as e:
        print(f"Geocoding failed for {place}: {e}")
    return None, None

# Apply geocoding to the 'Name' column
latitudes, longitudes = [], []
for suburb in info['Name']:
    lat, lon = get_coords(suburb)
    latitudes.append(lat)
    longitudes.append(lon)
    time.sleep(1)  # polite pause to avoid hitting API limits

info['latitude'] = latitudes
info['longitude'] = longitudes

In [5]:
missing = info[info["latitude"].isna() | info["longitude"].isna()]
missing["Name"].unique()
#Dee Why West doesn't exist as a suburb polygon, so will need to change the name to Dee Why

array(['Dee Why West'], dtype=object)

In [6]:
dee_why_west_lat = -33.73441
dee_why_west_lon = 151.28278
#Found the lat/lon information online

In [7]:
info.loc[info["Name"] == "Dee Why West", "latitude"] = dee_why_west_lat
info.loc[info["Name"] == "Dee Why West", "longitude"] = dee_why_west_lon
#inputting lat and lon from online into info

In [8]:
info.head()

,energy_asset,Name,Area,Dwellings,Persons,Residential,Commercial,Industrial,Primary Production,Education,Hospital/Medical,Transport,Parkland,Water,Other,latitude,longitude
BLAKE,AG_BLAKEHURST,Blakehurst,7,10081,28521,0.850,0.005,0.021,0.0,0.022,0.001,0.000,0.102,0.0,0.0,-33.989852,151.108560
PUNCH,AG_PUNCHBOWL,Punchbowl,9,17514,50395,0.826,0.048,0.038,0.0,0.025,0.002,0.001,0.061,0.0,0.0,-33.928717,151.052259
MEADO,AG_MEADOWBANK,Meadowbank,15,22420,56948,0.825,0.023,0.015,0.0,0.036,0.014,0.006,0.082,0.0,0.0,-33.817499,151.088425
MOSMA,AG_MOSMAN,Mosman,11,25967,52831,0.817,0.046,0.001,0.0,0.013,0.001,0.000,0.123,0.0,0.0,-33.828354,151.247685
RIVER,AG_RIVERWOOD,Riverwood,10,12528,34547,0.813,0.018,0.042,0.0,0.035,0.002,0.001,0.089,0.0,0.0,-33.947222,151.053056


# Creating new csv file
- One row per day
- 30 days before the holiday
- the holiday itself
- 30 days after the holiday
- So 61 rows per holiday × year × station
- also ensures new year's day and christmas day include any days within the 30 +/- days that are not in the same year
- includes columns defining what the day's name is (Monday, Tuesday etc), and if it's a weekend (True/False)
----------------------------------
- For each day, the function will compute:
- Block‑level mean, standard deviation, and variance of the relative‑rank values for that day’s 24‑hour profile.
---------------------------------
- Using new time blocks:
- 04–10
- 10–15
- 15–20
- 20–24
- 00–04
---------------------------------
- Using your 2‑year forward‑looking ranking window (Y + Y+1)
- metadata integrated (station id, station name, residential fraction, industrial fraction, dwellings, persons)

** UPDATE 10_03_26
- now includes lat and lon columns of the approximate location of each substation (based on suburb location)
- created a new column for 2 'new' public holidays, which combine pre-existing holidays as they are very similar. All easter related holidats => Easter Long Weekend, Christmas Day and Boxing Day => Christmas and Boxing Day

** UPDATE 16_03_26
- actually calculated the mean demand relative rank for each time block on the holiday groups
- maintains the WIDE formatting of time block columns (00_04_mean etc)

In [ ]:
def compute_two_year_daily_relative_rank_csv(
    demand,
    holiday_lib,
    info,
    holiday_groups=None,
    window_days=30,
    blocks=None,
    out_csv="full_nsw_relative_rank_wide.csv"
):
    """
    Compute daily block-level mean, std, and variance of relative ranks
    for ±window_days around each holiday (original + grouped),
    for each year and station.

    Output = WIDE FORMAT:
        one row per station × holiday_group × year × date
        with columns like 00_04_mean_relative_rank, 04_10_mean_relative_rank, etc.
    """

    import pandas as pd
    import numpy as np
    from tqdm import tqdm

    if blocks is None:
        raise ValueError("You must supply a dictionary of time blocks.")

    if holiday_groups is None:
        holiday_groups = {}

    # Convert demand index to datetime
    demand.index = pd.to_datetime(demand.index)

    # Hourly mean demand
    hourly = demand.resample("h").mean()

    # All station columns
    stations = [c for c in hourly.columns if c not in ["date", "hour"]]

    years = list(range(2004, 2018))

    # Build unified holiday dictionary
    all_holidays = dict(holiday_lib)
    for original, group in holiday_groups.items():
        if group not in all_holidays:
            all_holidays[group] = holiday_lib[original]

    # Map block names to weather-style names
    block_map = {
        "00-04": "00_04",
        "04-10": "04_10",
        "10-15": "10_15",
        "15-20": "15_20",
        "20-24": "20_24"
    }

    rows = []

    total_iterations = len(all_holidays) * len(years) * len(stations)
    pbar = tqdm(total=total_iterations, desc="Computing wide-format demand ranks")

    # MAIN LOOP
    for holiday_name, holiday_func in all_holidays.items():
        for year in years:

            # Holiday reference date
            ref_date = holiday_func(year)

            # Expanded pool for spillover
            pool_start = pd.Timestamp(f"{year}-01-01") - pd.Timedelta(days=31)
            pool_end   = pd.Timestamp(f"{year+1}-12-31") + pd.Timedelta(days=31)
            expanded_pool = hourly.loc[pool_start:pool_end]

            if expanded_pool.empty:
                pbar.update(len(stations))
                continue

            # Extract ±window_days
            win_start = ref_date - pd.Timedelta(days=window_days)
            win_end   = ref_date + pd.Timedelta(days=window_days)
            window = expanded_pool.loc[win_start:win_end].copy()

            if window.empty:
                pbar.update(len(stations))
                continue

            window["date"] = window.index.date
            window["hour"] = window.index.hour

            # Ranking pool = Y + Y+1
            rank_start = pd.Timestamp(f"{year}-01-01")
            rank_end   = pd.Timestamp(f"{year+1}-12-31")
            ranking_pool = hourly.loc[rank_start:rank_end]

            # LOOP STATIONS
            for station in stations:
                pbar.update(1)

                if station not in window.columns:
                    continue

                # Compute relative rank per hour
                rp = ranking_pool[[station]].copy()
                rp["date"] = rp.index.date
                rp["hour"] = rp.index.hour

                rp["rank"] = rp.groupby("hour")[station].rank(method="average")
                n_days = rp.groupby("hour")["date"].transform("nunique")
                rp["relative_rank"] = rp["rank"] / n_days

                # Merge relative rank into window
                W = window.merge(
                    rp[["date", "hour", "relative_rank"]],
                    on=["date", "hour"],
                    how="left"
                )

                # Build one row per date
                for d in sorted(W["date"].unique()):
                    day_slice = W[W["date"] == d]

                    row = {
                        "station_code": station,
                        "holiday_group": holiday_name,
                        "year": year,
                        "date": d
                    }

                    # Compute block-level stats directly into wide columns
                    for block_name, hours in blocks.items():
                        h_start, h_end = hours
                        mask = (day_slice["hour"] >= h_start) & (day_slice["hour"] < h_end)
                        vals = day_slice.loc[mask, "relative_rank"]

                        wide_block = block_map[block_name]

                        row[f"{wide_block}_mean_relative_rank"] = vals.mean()
                        row[f"{wide_block}_std_relative_rank"]  = vals.std()
                        row[f"{wide_block}_var_relative_rank"]  = vals.var()

                    rows.append(row)

    pbar.close()

    # Build DataFrame
    df = pd.DataFrame(rows)
    
    # --- Add holiday/weekend metadata columns ---
    df["date"] = pd.to_datetime(df["date"])
    
    df["is_holiday"] = df.apply(
        lambda r: r["date"] == all_holidays[r["holiday_group"]](r["year"]),
        axis=1
    )
    
    df["weekday_name"] = df["date"].dt.day_name()
    df["is_weekend"] = df["weekday_name"].isin(["Saturday", "Sunday"])
    
    # Merge metadata
    df = df.merge(info, left_on="station_code", right_index=True, how="left")
    
    # Save
    df.to_csv(out_csv, index=False)
    
    return df


In [ ]:
# Time blocks
blocks = {
    "00-04": (0, 4),
    "04-10": (4, 10),
    "10-15": (10, 15),
    "15-20": (15, 20),
    "20-24": (20, 24),
}


In [ ]:
out_path = "/home/565/pv3484/aus_substation_electricity/data/cleaned_data/full_nsw_relative_rank.csv"

demand_rank = compute_two_year_daily_relative_rank_csv(
    demand=demand,
    holiday_lib=HOLIDAYS_VIC,
    info=info,
    holiday_groups=HOLIDAY_GROUPS,
    window_days=30,
    blocks=blocks,
    out_csv=out_path
)


# Trouble shooting

In [9]:
import pandas as pd

demand_rank = pd.read_csv("/home/565/pv3484/aus_substation_electricity/data/cleaned_data/full_nsw_relative_rank.csv")

In [10]:
demand_rank.columns.tolist()


['station_code',
 'holiday_group',
 'year',
 'date',
 '00_04_mean_relative_rank',
 '00_04_std_relative_rank',
 '00_04_var_relative_rank',
 '04_10_mean_relative_rank',
 '04_10_std_relative_rank',
 '04_10_var_relative_rank',
 '10_15_mean_relative_rank',
 '10_15_std_relative_rank',
 '10_15_var_relative_rank',
 '15_20_mean_relative_rank',
 '15_20_std_relative_rank',
 '15_20_var_relative_rank',
 '20_24_mean_relative_rank',
 '20_24_std_relative_rank',
 '20_24_var_relative_rank',
 'is_holiday',
 'weekday_name',
 'is_weekend',
 'energy_asset',
 'Name',
 'Area',
 'Dwellings',
 'Persons',
 'Residential',
 'Commercial',
 'Industrial',
 'Primary Production',
 'Education',
 'Hospital/Medical',
 'Transport',
 'Parkland',
 'Water',
 'Other',
 'latitude',
 'longitude']